# Classification Validation

This notebook documents the creation adn validation of the employee-oriented and product-oriented video datasets used in the submission. The videos were extracted from teh provided youtube 2016 dataset using aggregation pipelines in MongoDB Compass and exported for further analysis and validation in Python.

### Tag Extraction

For the initial step, I extracted the snippet.tags that were present in the videos to analyze whether it would yield a valuable insight on the topic of the vidoes. To be specific, whether the video focuses primarily on employer branding and recruitment topics or on products, services and technology.:

```javascript
[
  { "$unwind": "$snippet.tags" },
  {
    "$group": {
      "_id": { "$toLower": "$snippet.tags" },
      "count": { "$sum": 1 }
    }
  },
  { "$sort": { "count": -1 } }
]
```
This aggregation was used to identify commonly occurring tags in the dataset. Based on these results, keyword dictionaries for employee-oriented and product-oriented content were created.

## Classification

### Employee-based Classification 

To receive the employee classification, videos containing employee-related keywords such as jobs, careers, internships, HR, employer brandings, etc. were extracted. For this, the followign aggregation was used:

```javascript
[
  {
    $match: {
      "snippet.tags": {
        $elemMatch: {
          $regex:
            "employee|employees|career|careers|job|jobs|work|team|hr|human resources|internship|internships|recruiting|karriere|mitarbeiter|ausbildung|lehrstelle|arbeit|emploi|lavoro|trabajo|employer branding",
          $options: "i"
        }
      }
    }
  },
  {
    $project: {
      company: 1,
      channelId: 1,
      title: "$snippet.title",
      description: "$snippet.description",
      tags: "$snippet.tags",
      publishedAt: "$snippet.publishedAt",
      views: "$statistics.viewCount",
      likes: "$statistics.likeCount",
      comments: "$statistics.commentCount"
    }
  }
]
```
This aggregation resulted in 3120 videos, which should suffice to continue with the analysis and make a conclusion for ,y hypothesis. This data needed for further analysis and validation was exported in a json file named `employee-based-content.json`.

### Product-based Classification
For the product-based classification, a similar extraction was performed, but with different tags. The following aggregation was used:

```javascript
[
  {
    "$match": {
      "snippet.tags": {
        "$elemMatch": {
          "$regex": "product|products|service|services|innovation|technology|software|app|application|tutorial|how to|demo|installation|commercial|advertisement|advertising|spot|tv spot|werbung|brand|machine|system|solution",
          "$options": "i"
        }
      }
    }
  },
  {
    "$project": {
      "company": 1,
      "channelId": 1,
      "title": "$snippet.title",
      "description": "$snippet.description",
      "tags": "$snippet.tags",
      "publishedAt": "$snippet.publishedAt",
      "views": "$statistics.viewCount",
      "likes": "$statistics.likeCount",
      "comments": "$statistics.commentCount"
    }
  }
]
```
This aggregation resulted into 4605 videos extracted. The data was again exported in a json file named `product-based-content.json`.


# Validation

In order to validate the results, I will load the two datasets and manually examine a few random entries to ensure they fit their classification. 

Note: I used Python version 3.10.7

In [2]:
import pandas as pd

employee = pd.read_json("employee-based-content.json")
product = pd.read_json("product-based-content.json")

### Dataset Overview


In this section we will get an overview of the two exported datasets. This includes the size, an overview of the elements in the video and classification verification.

Now let's take a look at the number of videos in each dataset.

In [3]:
print("Employee Data size: ", len(employee))
print("Product Data size: ", len(product))

Employee Data size:  3120
Product Data size:  4605


In [4]:
employee.head()

,_id,channelId,company,title,description,tags,publishedAt,views,likes,comments
0,jiHmzKeqeT8,UCLAKVzA5WEadvys9CKOksQg,Adecco SA,Klåss Clerkx - CEO for One Month 2016 at Adecc...,The #CEO1Month selection process started with ...,"[CEO1MONTH, Adecco Way to Work, bootcamp]",2016-05-25T15:51:58.000Z,156,1,0
1,1p9NjGCEieg,UCrSSdGg-aH_HgCHtsnoonIg,Adecco SA,Adecco Thailand WayToWork 2014,กะเทาะเปลือกเด็กจบใหม่ ครั้งที่ 2 กิจกรรมเพื่อ...,"[Way2WorkTh, AdeccoThailand, WayToWork, HRtwt,...",2014-04-30T11:10:20.000Z,1249,5,1
2,bmvCxO3zvsQ,UCrSSdGg-aH_HgCHtsnoonIg,Adecco SA,Adecco Thailand Career Up - Marketing Jobs,How to Introduce Yourself in English for Marke...,"[Marketing, Adecco, Jobs Seeker, Jobs]",2011-09-23T03:24:05.000Z,4535,6,3
3,61taXGTki5g,UCLAKVzA5WEadvys9CKOksQg,Adecco SA,Jobs bij TNT: Customer Service medewerkers in ...,"Je bent klantvriendelijk, goed in communicatie...","[Jobs, TNT, Customer Service, Machelen, België]",2016-05-17T11:15:11.000Z,131,1,0
4,W33uEUbULRs,UC7wvKWegeU9FGzEcVhJVoDw,ABB Ltd,Anna – Success begins with you,"Success begins with you because, at ABB, you c...","[ABB Ltd, ABB Careers, Technology, Engineering]",2016-07-18T14:27:22.000Z,379,5,1


In [5]:
product.head()

,_id,channelId,company,title,description,tags,publishedAt,views,likes,comments
0,EvjO6DIurLk,UCXyz4vATE49KlC_7vUY-bdw,Cembra Money Bank AG,Cembra Money Bank Famiglia con divano Italiano,Rebranding: Campagna Autunno 2013,"[Cembra Money Bank, Familie, Sofa, GE Money Ba...",2013-11-11T10:26:04.000Z,27387,0,0
1,jMlOv0Z5GpE,UCJ8IXb4xgt85gSKMpxXfpVQ,Bossard Holding AG,So funktioniert die Wärme-Einbettung eines Tap...,In warmem Kunststoff feste Verbindungen schaff...,"[kvt, fastening, kvt-fastening, Verbindungsele...",2015-06-22T07:47:56.000Z,293,{'$numberDouble': 'NaN'},{'$numberDouble': 'NaN'}
2,61taXGTki5g,UCLAKVzA5WEadvys9CKOksQg,Adecco SA,Jobs bij TNT: Customer Service medewerkers in ...,"Je bent klantvriendelijk, goed in communicatie...","[Jobs, TNT, Customer Service, Machelen, België]",2016-05-17T11:15:11.000Z,131,1,0
3,W33uEUbULRs,UC7wvKWegeU9FGzEcVhJVoDw,ABB Ltd,Anna – Success begins with you,"Success begins with you because, at ABB, you c...","[ABB Ltd, ABB Careers, Technology, Engineering]",2016-07-18T14:27:22.000Z,379,5,1
4,jOWd1QNgr7c,UCf6fYOaOmIGuLPSGncxELNA,Baloise Holding AG,Baloise Jobcast #4: Personal Branding,Wie präsentiere ich meine Eigenmarke? In diese...,"[bewerben, auftritt, image, baloise, podcast, ...",2016-05-02T14:47:55.000Z,47,1,0


In [8]:
print("Number of unique companies in Employee Data: ", employee["company"].nunique())
print("Number of unique companies in Product Data: ", product["company"].nunique())

Number of unique companies in Employee Data:  94
Number of unique companies in Product Data:  110


In [9]:
employee["company"].value_counts().head(10)

company
Adecco SA                 509
Nestle SA                 332
Swatch Group AG/The       155
Novartis AG               139
Roche Holding AG          137
Helvetia Holding AG       129
UBS Group AG              119
Credit Suisse Group AG    113
ABB Ltd                   112
Vontobel Holding AG       112
Name: count, dtype: int64

In [10]:
product["company"].value_counts().head(10)

company
Nestle SA                         676
ABB Ltd                           413
Helvetia Holding AG               233
Roche Holding AG                  222
UBS Group AG                      206
Georg Fischer AG                  204
Logitech International SA         135
Novartis AG                       131
Credit Suisse Group AG            130
Sunrise Communications Group A    124
Name: count, dtype: int64

The dataset contains videos from a large number of companies. However, several companies such as Nestlé SA, Adecco SA and ABB Ltd contribute a disproportionately large number of videos. This concentration should be considered when interpreting later results.

### Time Range

Now let's derive the timeframe of these videos. The goal is to know when these videos were published. Based on the output, the videos were published between 2007 and 2024.

In [13]:
employee["publishedAt"] = pd.to_datetime(employee["publishedAt"], format='mixed')
product["publishedAt"] = pd.to_datetime(product["publishedAt"], format='mixed')

In [29]:
print("Employee Data publishedAt range: ", employee["publishedAt"].min().strftime('%Y-%m-%d'), " to ", employee["publishedAt"].max().strftime('%Y-%m-%d'))
print(employee["publishedAt"].dt.year.value_counts().sort_index())

Employee Data publishedAt range:  2007-12-10  to  2024-03-28
publishedAt
2007      1
2008      1
2009     16
2010    131
2011    256
2012    407
2013    566
2014    641
2015    641
2016    427
2018      1
2019      1
2020      1
2021      1
2022     12
2023     13
2024      4
Name: count, dtype: int64


In [30]:
print("Product Data publishedAt range: ", product["publishedAt"].min().strftime('%Y-%m-%d'), " to ", product["publishedAt"].max().strftime('%Y-%m-%d'))
print(product["publishedAt"].dt.year.value_counts().sort_index())

Product Data publishedAt range:  2007-11-19  to  2022-12-01
publishedAt
2007       1
2008       5
2009      75
2010     174
2011     413
2012     470
2013     800
2014     951
2015    1008
2016     701
2022       7
Name: count, dtype: int64


In [27]:
print("Number of missing values in Employee Data: ", employee.isna().sum().sum())
print("Number of missing values in Employee Data by column: ")
print(employee.isna().sum())
print("\n")
print("Number of missing values in Product Data: ", product.isna().sum().sum())
print("Number of missing values in Product Data by column: ")
print(product.isna().sum())

Number of missing values in Employee Data:  97
Number of missing values in Employee Data by column: 
_id             0
channelId      33
company        33
title           0
description     0
tags            0
publishedAt     0
views           0
likes           3
comments       28
dtype: int64


Number of missing values in Product Data:  21
Number of missing values in Product Data by column: 
_id            0
channelId      7
company        7
title          0
description    0
tags           0
publishedAt    0
views          0
likes          0
comments       7
dtype: int64


The datasets contain missing values primarily in the engagement metrics (likes and comments). Missing company and channel identifiers are limited to 40 observations and represent less than 1% of the data.

### Overlap Analysis



In [32]:
employee_ids = set(employee["_id"])
product_ids = set(product["_id"])

overlap = employee_ids.intersection(product_ids)

print("The number of overlapping IDs in the two datasets is: ", len(overlap))

The number of overlapping IDs in the two datasets is:  956


To evaluate the quality of the keyword-based classification, the overlap between the employee-oriented and product-oriented datasets was examined.

A total of **956 videos** appear in both datasets. This indicates that these videos contain keywords associated with both employee-oriented and product-oriented content and were therefore classified into both categories during the extraction process.

Given that the employee-oriented dataset contains 3,120 videos and the product-oriented dataset contains 4,605 videos, the overlap represents approximately **30.6% of the employee-oriented videos** and **20.8% of the product-oriented videos**.

This result is not unexpected, as corporate videos can simultaneously discuss employees, careers, workplace culture, products, services, or technology. For example, videos about internship programs related to a company's technology, employee testimonials discussing products, or recruitment campaigns promoting innovative products may contain keywords from both categories.

The overlapping videos will be examined using a random sample. The goal is to identify common characteristics of videos that were assigned to both categories and to assess whether the keyword dictionaries require further refinement.

In [34]:
overlap_df = employee[employee["_id"].isin(overlap)]

overlap_sample = overlap_df.sample(30, random_state=42)

overlap_sample[["title", "company", "tags"]]

,title,company,tags
1347,Warehouse + NDIC Supply Chain | What You Do Ma...,Nestle SA,"[Nestle, Nestle USA, Nestle on Youtube, Nestle..."
2912,Straumann® CARES® - Wax Abutment,Straumann Holding AG,"[Wax abutment, Cares Visual, abutment, abutmen..."
1819,Iba Na – Carbonara | NESTLÉ All Purpose Cream ...,Nestle SA,"[nestle, nestle philippines, nestle ph, nestle..."
1866,Arun Sundararajan on how work will be transformed,Swiss Re AG,"[professor, New York University, Leonard N. St..."
2263,Simone talks about life at SGS,SGS SA,"[testimonial, employer branding, video, sgs pe..."
640,Découvrez le nouveau site du groupe Adecco France,Adecco SA,"[Adecco, groupe Adecco France, emploi, recrute..."
938,PHASETREAT® Demulsifiers - An ecofriendly solu...,Clariant AG,"[demulsifiers, oil, innovation, Clariant (Busi..."
787,"Moving from ""Just in Case"" to ""Just in Time"" i...",Ascom Holding AG,"[Ascom, Ascom Network Testing, TEMS, TEMS Solu..."
379,New Launcher with TEMS Investigation – Fast ac...,Ascom Holding AG,"[Ascom, Ascom Network Testing, TEMS, TEMS Inve..."
1133,ABB motion control products - Product Synchron...,ABB Ltd,"[ABB, motion control, motion control products,..."


### Removing overlapping videos

The overlap analysis showed that 956 videos appeared in both the employee-oriented and product-oriented datasets. Since these videos contain keywords from both classification groups, they are ambiguous for the purpose of a direct comparison.

To avoid double-counting and to make the later engagement analysis cleaner, these overlapping videos are removed from the two main datasets. The final comparison will therefore be based only on videos that were uniquely classified as either employee-oriented or product-oriented.

In [35]:
employee_ids = set(employee["_id"])
product_ids = set(product["_id"])

overlap_ids = employee_ids.intersection(product_ids)

employee_clean = employee[~employee["_id"].isin(overlap_ids)].copy()
product_clean = product[~product["_id"].isin(overlap_ids)].copy()

print("Original employee-oriented videos:", len(employee))
print("Original product-oriented videos:", len(product))
print("Overlapping videos removed:", len(overlap_ids))
print("Clean employee-oriented videos:", len(employee_clean))
print("Clean product-oriented videos:", len(product_clean))
print("Total clean dataset:", len(employee_clean) + len(product_clean))

Original employee-oriented videos: 3120
Original product-oriented videos: 4605
Overlapping videos removed: 956
Clean employee-oriented videos: 2164
Clean product-oriented videos: 3649
Total clean dataset: 5813


After removing overlapping videos, the final dataset contains 2,164 employee-oriented videos and 3,649 product-oriented videos. This results in a clean comparison dataset of 5,813 videos. The overlap group is excluded from the main analysis because its classification is ambiguous.